In [1]:
!pip install datasets transformers tokenizers accelerate fasttext

Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.4-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 5.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 11.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 48.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 74.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 107.4 MB/s  0:00:00m0:00:0100:01
Using cached pybind11-3.0.4-py3-none-any.whl (314 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.8/799.8 kB 11.2 MB/s  0:00:00
  Created wheel for fasttext: filename=fasttext-0.9.3-cp311-cp311-linux_x86_64.whl size=343267 sha256=c10416ab054b5ea5bfa6

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "allenai/c4",
    "sv",
    split="train",
    streaming=True
)

for i, sample in enumerate(dataset):
    print(sample["text"][:200])
    if i > 3:
        break

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Zara's Custom Tailor (Pattaya, Thailand) - omdömen
Restauranger i närheten av Zara's Custom Tailor
Saker att göra i närheten av Zara's Custom Tailor
Zara's Custom Tailor, Pattaya - omdömen
Nr 16 av 53
Hej Aidan!
Du placerade dig som nummer ett i PP1U, men blev dock inte svensk mästare då du rider för Great Britain.
-Det kändes ganska bra. Jag hade absolut inte förväntat mig det. Men han var på gång
Vis – en liten rofylld pärla - Reseguiden
Vis – en liten rofylld pärla
Reseguidens medlem Berit, alias "Desires" återvänder till Kroatien gång på gång. Nu tar hon oss med till sin favorit, den ljuvlig
Volvo 245 DL -85 som ska räddas från gubbdöden! - Sidan 7 - Garaget
19 702 Visningar
Dom blev fina fälgarna bra jobbat
20 juli · 1 374 Inlägg
Nästa dag i garaget vart det bara på det igen! Började med
﻿ 330 Mudclaw Fell Shoes At Running W9Y2eEIbDH
Merrell Men's ShoesFolkstone Running Vapor Eastern Trail Glove 2 Fc1TJKl
Shoes Summer Aj4115 Zoom Nike Size White Turbo Gauze Pegasus Running IE2WDH9

In [5]:
import fasttext
import urllib.request

# download model
urllib.request.urlretrieve(
    "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin",
    "lid.176.bin"
)

model = fasttext.load_model("lid.176.bin")

def is_sv_or_en(text):
    pred = model.predict(text[:500])
    lang = pred[0][0].replace("__label__", "")
    return lang in ["sv", "en"]

In [11]:
import json
import random
import re

TARGET_SIZE = 20000
SKIP_PROB = 0.99
BATCH_SIZE = 100

data = []

# ⚡ Fast heuristic filter (cheap)
def fast_lang_filter(text):
    if not isinstance(text, str) or len(text) < 100:
        return False

    text = text.lower()

    # English signals
    en_hits = sum(word in text for word in ["the", "and", "is", "are", "to"])

    # Swedish signals
    sv_hits = sum(char in text for char in "åäö") + \
              sum(word in text for word in ["och", "det", "att", "som"])

    return (en_hits >= 2) or (sv_hits >= 1)


# 🧹 Cleaning function
def clean_text(text):
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


batch = []

for i, sample in enumerate(dataset):

    # 🎲 Skip most data (sampling)
    if random.random() < SKIP_PROB:
        continue

    text = sample.get("text", "")

    # ⚡ VERY FAST filter (no ML)
    if not fast_lang_filter(text):
        continue

    # Clean minimal for model
    text = text.replace("\n", " ").replace("\r", " ").strip()

    batch.append(text[:500])  # fastText doesn't need full text

    # 🚀 Batch prediction
    if len(batch) == BATCH_SIZE:
        try:
            preds, probs = model.predict(batch)

            for t, p, pr in zip(batch, preds, probs):
                lang = p[0].replace("__label__", "")

                if lang in ["sv", "en"] and pr[0] > 0.8:
                    cleaned = clean_text(t)

                    if len(cleaned) > 100:
                        data.append({"text": cleaned})

        except Exception as e:
            # Skip batch if something breaks
            pass

        batch = []

    # 🎯 Stop early when target reached
    if len(data) >= TARGET_SIZE:
        break

    if i % 5000 == 0:
        print(f"Processed {i}, collected {len(data)}")


# 💾 Save output
with open("data.jsonl", "w", encoding="utf-8") as f:
    for row in data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("✅ Final dataset size:", len(data))

Processed 145000, collected 1094
Processed 1335000, collected 10548
✅ Final dataset size: 20067


In [4]:
import os
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer()

tokenizer.train(
    files=["data.jsonl"],
    vocab_size=16000,
    min_frequency=2
)

# ✅ Create directory
os.makedirs("tokenizer", exist_ok=True)

# ✅ Save model
tokenizer.save_model("tokenizer")

['tokenizer/vocab.json', 'tokenizer/merges.txt']

In [4]:
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size=16000,
    n_layer=4,
    n_head=4,
    n_embd=256
)

model = GPT2LMHeadModel(config)

In [5]:
from tokenizers import ByteLevelBPETokenizer

# Load trained tokenizer
tokenizer = ByteLevelBPETokenizer(
    "tokenizer/vocab.json",
    "tokenizer/merges.txt"
)

In [6]:
from datasets import load_dataset
from transformers import Trainer, TrainingArguments
import torch

# Load dataset
dataset = load_dataset("json", data_files="data.jsonl")

# Max sequence length
MAX_LEN = 128

# Tokenization with padding + labels
def tokenize(example):
    ids = tokenizer.encode(example["text"]).ids[:MAX_LEN]
    padding_length = MAX_LEN - len(ids)

    input_ids = ids + [0] * padding_length
    attention_mask = [1] * len(ids) + [0] * padding_length

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.copy()
    }

# Apply tokenization
tokenized = dataset.map(tokenize, remove_columns=["text"])

# Set format for PyTorch
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Training config
training_args = TrainingArguments(
    output_dir="./model",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"]
)

# Train
trainer.train()


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,9.482948
20,9.118331
30,9.217615
40,8.999432
50,8.898198
60,8.594677
70,8.749100
80,8.333842
90,8.544470
100,8.459284


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10034, training_loss=6.794174016413043, metrics={'train_runtime': 134.3592, 'train_samples_per_second': 149.353, 'train_steps_per_second': 74.68, 'total_flos': 48693296627712.0, 'train_loss': 6.794174016413043, 'epoch': 1.0})

In [11]:
trainer.save_model("model/final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
import os

for root, dirs, files in os.walk("model"):
    print(root, files)

model []
model/final ['model.safetensors', 'training_args.bin', 'config.json', 'generation_config.json']
model/checkpoint-10000 ['scheduler.pt', 'model.safetensors', 'training_args.bin', 'rng_state.pth', 'config.json', 'trainer_state.json', 'optimizer.pt', 'generation_config.json']
model/checkpoint-10034 ['scheduler.pt', 'model.safetensors', 'training_args.bin', 'rng_state.pth', 'config.json', 'trainer_state.json', 'optimizer.pt', 'generation_config.json']


In [14]:
from transformers import GPT2LMHeadModel
from tokenizers import ByteLevelBPETokenizer
import torch

# Load tokenizer
tokenizer = ByteLevelBPETokenizer(
    "tokenizer/vocab.json",
    "tokenizer/merges.txt"
)

# ✅ Load correct model
model = GPT2LMHeadModel.from_pretrained("model/final")
model.eval()

# Swedish prompt 🇸🇪
prompt = "Det var en gång"

input_ids = torch.tensor([tokenizer.encode(prompt).ids])

output = model.generate(
    input_ids,
    max_length=100,
    do_sample=True,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2
)

print(tokenizer.decode(output[0].tolist()))

Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Det var en gång för att göra i de inte på den det om vara en jag? Vi har mycket bra. Jag ska du var man på något jag går vi. Jag här ju till att jag är ett alla så kan nu så när att så finns med det var bara om det kan man!! Men du lite mer och som är jag är han är så som gör, och på att jag kan det jag är jag inte det också jag av att då man jag när jag får även inte.


In [8]:
ls model/

checkpoint-10000/  checkpoint-10034/


In [9]:
model = GPT2LMHeadModel.from_pretrained("./model/checkpoint-10000")

Loading weights:   0%|          | 0/52 [00:00<?, ?it/s]